# Value: QKD vs PQC (bands, not exact dollars)

*Secure the Grid · Step 3, Part 5 + what we learned*

We are not quoting a vendor invoice. We put costs in **relative units** (order-of-magnitude bands from DOE/EPB-era figures and engineering judgment — see `data/cost_bands.json`). Then we measure what each portfolio **buys in security** with the same attack rules as notebook 4.

**Questions:** What does one QKD link cost next to a PQC update? What do we get for the difference? Where does quantum help today, and where does it not?

In [ ]:
# Use the qBraid-SDK (v0.12.2) kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: this one has no " + " or ".join(missing) + ". Click the kernel name "
                      "at the top right of the notebook, pick qBraid-SDK (v0.12.2), then run this cell again.")
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

from pathlib import Path
import sys
_here = Path.cwd().resolve()
for _cand in (_here, _here.parent):
    if (_cand / "secure_grid").is_dir():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from secure_grid import value_experiment as vex

bands = vex.load_bands()
cmp_ = vex.compare_portfolios()
print(bands["about"])
print("Unit:", bands["unit"], "—", bands["note"][:120], "...")

## Cost of one QKD link vs one PQC upgrade (bands)

Mid band (lo–hi in the JSON):

In [ ]:
one = pd.DataFrame([
    {"item": "PQC software/firmware on one link",
     "lo": bands["pqc_per_link"]["lo"], "mid": bands["pqc_per_link"]["mid"], "hi": bands["pqc_per_link"]["hi"]},
    {"item": "QKD equipment (Alice+Bob pair)",
     "lo": bands["qkd_per_link"]["equipment_pair"]["lo"],
     "mid": bands["qkd_per_link"]["equipment_pair"]["mid"],
     "hi": bands["qkd_per_link"]["equipment_pair"]["hi"]},
    {"item": "Dedicated fiber / channel (add-on)",
     "lo": bands["qkd_per_link"]["dedicated_fiber"]["lo"],
     "mid": bands["qkd_per_link"]["dedicated_fiber"]["mid"],
     "hi": bands["qkd_per_link"]["dedicated_fiber"]["hi"]},
    {"item": "Trusted node (only if distance needs it)",
     "lo": bands["qkd_per_link"]["trusted_node_if_needed"]["lo"],
     "mid": bands["qkd_per_link"]["trusted_node_if_needed"]["mid"],
     "hi": bands["qkd_per_link"]["trusted_node_if_needed"]["hi"]},
])
print(one.to_string(index=False))
qkd_mid = vex.cost_qkd_link(bands, needs_trusted_node=False, which="mid")
print(f"\nOne metro QKD link (equip+fiber, mid): {qkd_mid:.0f} units")
print(f"One PQC link upgrade (mid):            {vex.cost_pqc_link(bands):.0f} units")
print(f"Ratio (mid): about {qkd_mid / vex.cost_pqc_link(bands):.0f}× a PQC upgrade")

## Experiment: same network, three portfolios

1. **Today** — illustrative legacy gaps  
2. **PQC only** — our design with zero QKD links  
3. **PQC + QKD** — PQC everywhere, QKD on the four critical links in `defenses.json`

In [ ]:
rows = []
for name, key in [("today", "today"), ("PQC only", "pqc_only"), ("PQC + QKD", "pqc_plus_qkd")]:
    block = cmp_[key]
    sec = block["security"]
    cost = block.get("cost_mid")
    rows.append({
        "portfolio": name,
        "attack successes": sec.get("attack_successes_design", sec.get("attack_successes_today")),
        "wire taps detected": sec.get("wire_taps_detected", "—"),
        "cost mid (units)": None if cost is None else round(cost["total"], 1),
        "of which QKD": None if cost is None else round(cost["qkd"], 1),
    })
# fix today row label
rows[0]["attack successes"] = cmp_["today"]["security"]["attack_successes_today"]
print(pd.DataFrame(rows).to_string(index=False))
d = cmp_["difference_mid"]
print(f"\nDifference (PQC+QKD minus PQC-only, mid band):")
print(f"  Extra cost:     {d['extra_cost_units']:.0f} units")
print(f"  Extra taps detected: {d['extra_taps_detected']} "
      f"(attack successes both zero: {d['attack_successes_same']})")
print("QKD links:", cmp_["pqc_plus_qkd"]["qkd_links"])

**What the utility gets for the difference:** On this network, PQC-only already drives attack successes to **0**. Adding QKD costs ~**100×** more per those links (mid band) and buys **tap detection** on the QKD fibers (here: 4 taps flagged). Encryption alone does not tell you someone is listening; QKD’s error alarm does.

## Where quantum helps today (spy fraction experiment)

We vary how often Eve taps and measure QBER on the local simulator. Abort line = 11%.

In [ ]:
curve = pd.DataFrame(vex.spy_fraction_curve())
print(curve.to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(curve["eve_fraction"], curve["qber"], "o-", color="tab:red", label="QBER")
ax.axhline(0.11, color="k", ls="--", label="11% abort")
ax.set_xlabel("fraction of qubits Eve measures")
ax.set_ylabel("error rate")
ax.set_title("Can a careful spy hide? (local Aer)")
ax.legend()
fig.tight_layout()
plt.show()
print("First fraction that crosses abort:", float(curve.loc[curve["abort"], "eve_fraction"].iloc[0]))

## Where quantum helps / does not — and what failed

In [ ]:
print(pd.DataFrame(vex.quantum_helps_table()).to_string(index=False))

**What failed / we did not do (and why):**

- **Exact dollar quotes** — vendors and fiber deals are not public; we used bands so the comparison stays honest.
- **Paid IonQ Forte** — credits; free emulator already showed spy ≫ clean.
- **QAOA link picker** — stretch; greedy “QKD on top critical links” is enough for this value question.
- **OpenGridWorks ≠ our 11-node graph** — GIF is an illustrative OT overlay, not every HIFLD line.